# Stage 5. Evaluation and Ablation

Notebook ini menghitung metrik klinis lengkap pada prediksi out-of-fold hasil stage sebelumnya (Bland-Altman, ROC/AUC dengan threshold Youden, PPV/NPV, Cohen's kappa severity), lalu menjalankan ablation komponen arsitektur (CSA, fusion attention, dual loss, demografi). Uji cross-dataset generalization tidak dilakukan karena palm hanya satu populasi, berbeda dari situs konjungtiva yang punya dua populasi (CP-AnemiC dan Eyes-Defy).

## Environment Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, auc, cohen_kappa_score, confusion_matrix
import matplotlib.pyplot as plt

from configs import paths
from src.common import eval as evaluation
from src.common import features, manifest as manifest_utils, train

output_dir = paths.outputs_dir("palm")
manifest = pd.read_csv(output_dir / "manifest.csv")
manifest = manifest[manifest["roi_precropped"]].reset_index(drop=True)
manifest = manifest_utils.assign_kfold(manifest, n_splits=5, seed=42)
oof = pd.read_csv(output_dir / "multitask_oof_full_fusion_tuned.csv")

# Part A. Deep Clinical Evaluation

## Bland-Altman Analysis for Hemoglobin Regression

Bias dan limits of agreement mengukur kesepakatan estimasi hemoglobin model terhadap Rad-67 sebagai ground truth.

In [ ]:
bland_altman = evaluation.bland_altman_stats(oof["hb_true"], oof["hb_pred"])
print("bias", round(bland_altman["bias"], 3))
print("limits of agreement", round(bland_altman["lower_limit"], 3), "sampai", round(bland_altman["upper_limit"], 3))

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(bland_altman["mean_value"], bland_altman["difference"], alpha=0.4, s=10)
ax.axhline(bland_altman["bias"], color="black", label="bias")
ax.axhline(bland_altman["lower_limit"], color="red", linestyle="--", label="limits of agreement")
ax.axhline(bland_altman["upper_limit"], color="red", linestyle="--")
ax.set_xlabel("Mean of true and predicted Hb")
ax.set_ylabel("Predicted minus true Hb")
ax.set_title("Bland-Altman Plot")
ax.legend()
plt.show()

## ROC Curve and Sensitivity-Prioritized Threshold

Threshold Youden dipilih karena skrining anemia mengutamakan sensitivitas tinggi supaya minim false negative.

In [ ]:
y_true = oof["anemic_true"].to_numpy()
y_prob = oof["anemic_prob"].to_numpy()
false_positive_rate, true_positive_rate, _ = roc_curve(y_true, y_prob)
roc_auc = auc(false_positive_rate, true_positive_rate)
youden = evaluation.youden_optimal_threshold(y_true, y_prob)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(false_positive_rate, true_positive_rate, label=f"AUC = {roc_auc:.3f}")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve, Anemia Classification")
ax.legend()
plt.show()

for label, threshold in [("default 0.5", 0.5), ("Youden", youden["threshold"])]:
    prediction = (y_prob >= threshold).astype(int)
    ppv_npv = evaluation.ppv_npv(y_true, prediction)
    print(label, "threshold", round(threshold, 3), "ppv", round(ppv_npv["ppv"], 3), "npv", round(ppv_npv["npv"], 3))

## Severity Confusion Matrix and Cohen's Kappa

Evaluasi tiga kelas severity (Non-Anemic, Mild, Moderate) yang tersedia pada dataset ini, hanya pada baris dengan label severity valid.

In [ ]:
severity_valid = oof["severity_true"] >= 0
severity_true = oof.loc[severity_valid, "severity_true"]
severity_pred = oof.loc[severity_valid, "severity_pred"]

kappa = cohen_kappa_score(severity_true, severity_pred)
print("Cohen's kappa severity", round(kappa, 3))
print(confusion_matrix(severity_true, severity_pred))

# Part B. Component Ablation

Setiap komponen dinonaktifkan satu per satu dari konfigurasi full fusion tuned untuk mengukur kontribusinya terhadap MAE dan akurasi klasifikasi.

In [ ]:
handcrafted = pd.read_csv(output_dir / "handcrafted_features.csv")
deep_embeddings = np.load(output_dir / "deep_embeddings.npy")
embedding_uids = pd.read_csv(output_dir / "deep_embeddings_uids.csv")["uid"].tolist()

ablation_configs = {
    "no_fusion_attention": dict(use_fusion_attention=False),
    "no_demographics": dict(use_demographics=False),
    "no_site_token": dict(use_site_token=False),
    "mse_only": dict(regression_loss="mse_only"),
}

ablation_rows = []
for name, overrides in ablation_configs.items():
    result = train.run_kfold(
        manifest, handcrafted, deep_embeddings, embedding_uids,
        n_splits=5, epochs=60, **overrides,
    )
    fold_metrics = result["fold_metrics"]
    ablation_rows.append({
        "configuration": name,
        "mae": fold_metrics["mae"].mean(),
        "accuracy": fold_metrics["accuracy"].mean(),
    })

ablation_table = pd.DataFrame(ablation_rows)
ablation_table.to_csv(output_dir / "ablation_table.csv", index=False)
ablation_table

## CSA Ablation

Backbone tanpa modul channel spatial attention dibandingkan dengan backbone CSA penuh, mengisolasi kontribusi modul atensi terhadap kualitas embedding deep.

In [ ]:
backbone_no_csa = features.EmbeddingBackbone(backbone_name="resnet18", use_csa=False)
deep_embeddings_no_csa, embedding_uids_no_csa = features.extract_deep_embeddings(manifest, model=backbone_no_csa)
result_no_csa = train.run_kfold(
    manifest, handcrafted, deep_embeddings_no_csa, embedding_uids_no_csa,
    n_splits=5, epochs=60,
)
no_csa_metrics = result_no_csa["fold_metrics"]
print("no CSA MAE", round(no_csa_metrics["mae"].mean(), 3), "accuracy", round(no_csa_metrics["accuracy"].mean(), 3))